# Retrieve wide, rerank narrow

*A second model re-reads the shortlist — and only helps once it has something to read.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omnifroodle/couchbase_notebooks/blob/main/notebooks/retrieval/03_retrieve_wide_rerank_narrow.ipynb)
[![Open in Codespaces](https://img.shields.io/badge/Open%20in%20Codespaces-2f363d?logo=github&logoColor=white)](https://codespaces.new/omnifroodle/couchbase_notebooks?quickstart=1)

**Claim.** Retrieval and ranking are different jobs: a wide cheap search, then an expensive model over the shortlist, orders results better than either half alone.
**Result.** Reranking on the product name alone changes nothing (nDCG@10 0.769 → 0.766). The same model, given each product's category path as well, reaches 0.794 and puts a relevant product first for 98% of queries — for about three extra seconds per 40 queries.
**Requires.** couchbase · local-embeddings
**Read** ~10 min · **Run** ~4 min · **Cost** $0.00 — no API calls, the reranker runs on your machine

[`retrieval/02`](02_which_parts_helped.ipynb) left two things unfinished.

**Recall was about 0.40 against a ceiling of 0.462.** The right products were mostly *there*, in
the top 50, and what separated the strategies was the order they came back in. That is the shape
of problem reranking exists for.

**The department filter was a bet on a classifier.** When the classifier was right it removed
distractors; when it was wrong the right products were *gone*, and no amount of good ranking
downstream could bring them back. A reranker spends the same uncertainty differently: it
reorders instead of removing, so being wrong costs a position rather than a product.

This notebook builds the reranker, scores it against the same 40 queries and the same human
judgements, and tests both claims. The first attempt does not work, which turns out to be the
most useful thing in here.

In [ ]:
# --- Setup. Works in a local checkout and on Colab. -------------------------
import os
import pathlib
import subprocess
import sys

# Cloned on Colab, where there is no local checkout. Override to test a fork.
REPO_URL = os.environ.get("CBNB_REPO_URL", "https://github.com/omnifroodle/couchbase_notebooks")

try:
    import cbnb
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    root = next((p for p in [here, *here.parents] if (p / "cbnb" / "__init__.py").exists()), None)
    if root is None:
        # Colab: clone the repo so the committed datasets come with it.
        subprocess.check_call(["git", "clone", "--depth", "1", "--quiet", REPO_URL, "cbnb-repo"])
        root = pathlib.Path("cbnb-repo").resolve()
    sys.path.insert(0, str(root))
    import cbnb

settings = cbnb.bootstrap(requires=["couchbase", "local-embeddings"], extras=["metrics"])

## 1. Two models that score the same pair differently

Everything so far has used a **bi-encoder**: the product is embedded once, ahead of time, the
query is embedded at search time, and relevance is the distance between two vectors that never
met. That independence is what makes search fast — 6,482 product vectors were computed long
before you typed anything, and the index only has to compare numbers.

It is also what the model gives up. Each side is compressed into a fixed vector before it knows
the other exists.

A **cross-encoder** makes the opposite trade. It takes the query and one product *together*, in
a single forward pass, and returns a relevance score. Nothing can be precomputed, so it cannot
search a corpus — scoring all 6,482 products for one query would take minutes. Over fifty
candidates it takes a moment, and it gets to read the query while reading the product.

| | job | cost |
| --- | --- | --- |
| **Couchbase search** | find fifty plausible products among thousands | one indexed query |
| **cross-encoder** | put the best of those fifty first | fifty model passes |

One consequence matters for everything below: **reranking fifty candidates cannot change
recall@50.** It reorders the same fifty products. If the right product is not on the shortlist,
the reranker never sees it. Retrieval sets the ceiling; reranking decides what happens beneath
it.

In [2]:
from cbnb.datasets import load_wands_eval_set

evalset = load_wands_eval_set()
print(evalset.summary())

40 queries, 6,482 products, 6,986 judgements (2,506 Exact)


## 2. The index from `retrieval/01`, rebuilt if it is not there

Same 6,482 products, same single Search index carrying text, vectors and keyword fields. If you
ran [`01`](01_building_hybrid_search.ipynb) or [`02`](02_which_parts_helped.ipynb), every
`ensure_*` call below finds what it needs and returns.

In [3]:
import pandas as pd

from cbnb.couchbase_io import (
    connect,
    ensure_collection,
    ensure_vector_index,
    upsert_docs,
    wait_for_index,
)
from cbnb.embeddings import Embedder

pd.set_option("display.max_colwidth", None)

BUCKET, SCOPE, COLLECTION = settings.cb_bucket, "hybrid_retrieval", "products"
INDEX = "products_hybrid"

products = evalset.products.copy()
products["category_hierarchy"] = products.category_hierarchy.fillna("")
products["department"] = products.category_hierarchy.str.split(" / ").str[0]
products["text"] = (products.product_name.fillna("") + " "
                    + products.product_class.fillna("")).str.strip()

embedder = Embedder()
vectors = embedder.encode(products.text.tolist())

cluster = connect(settings)
collection = ensure_collection(cluster, BUCKET, SCOPE, COLLECTION)
upsert_docs(collection, {
    str(row.product_id): {
        "type": COLLECTION,
        "product_id": int(row.product_id),
        "text": row.text,
        "product_name": row.product_name,
        "department": row.department,
        "category_path": row.category_hierarchy,
        "embedding": vector.tolist(),
    }
    for row, vector in zip(products.itertuples(index=False), vectors, strict=True)
}, progress=False)

ensure_vector_index(
    cluster,
    bucket_name=BUCKET, scope_name=SCOPE, collection_name=COLLECTION, index_name=INDEX,
    vector_field="embedding", dims=embedder.dims,
    text_fields=["text"], keyword_fields=["department", "category_path"],
)
wait_for_index(cluster, bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX,
               expected=len(products))

  indexed 6482/6482 (ready)   

  indexed 6482/6482 (ready)   

6482

## 3. One query, before and after

The retrieval half is the hybrid search from `01`, asked for a deep shortlist rather than a
screenful. The reranking half is three lines: load the cross-encoder, hand it the query and the
candidates, sort by what it says.

In [4]:
import couchbase.search as search

from cbnb.couchbase_io import hybrid_search
from cbnb.rerank import Reranker

WHERE = dict(bucket_name=BUCKET, scope_name=SCOPE, index_name=INDEX)
SHORTLIST = 50          # candidates retrieved, and the depth the reranker reads
CANDIDATES = 200        # vector candidates the index considers before cutting to k
FIELDS = ["text", "category_path", "department"]


def retrieve(query, k=SHORTLIST, prefilter=None):
    """Wide and cheap: one hybrid Search request."""
    return hybrid_search(cluster, **WHERE, vector_field="embedding",
                         query_vector=embedder.encode_one(query),
                         text_query=search.MatchQuery(query, field="text"),
                         k=k, num_candidates=CANDIDATES, fields=FIELDS, prefilter=prefilter)


reranker = Reranker()      # cross-encoder/ms-marco-MiniLM-L-6-v2, ~80MB, CPU is fine
print(reranker)

Reranker('cross-encoder/ms-marco-MiniLM-L-6-v2')


In [5]:
QUERY = "high weight capacity bunk beds"

candidates = retrieve(QUERY)
reranked = reranker.rank(QUERY, candidates)          # reads the `text` field only

pd.DataFrame({
    "hybrid, top 10": pd.Series([c["text"] for c in candidates[:10]], index=range(1, 11)),
    "after reranking": pd.Series([s.text for s in reranked[:10]], index=range(1, 11)),
    "": pd.Series([f"was #{s.was_rank}" for s in reranked[:10]], index=range(1, 11)),
}).rename_axis(columns=repr(QUERY))

'high weight capacity bunk beds',"hybrid, top 10",after reranking,
1,bunk bed,nautilus standard bunk bed,was #6
2,full bunk bed,full bunk bed,was #2
3,brina bunk bed,"full over full bunk bed with twin size trundle , pine wood bunk bed with guardrails for kids and teens",was #29
4,eruh bunk bed,eruh bunk bed,was #4
5,renan bunk bed,bunk bed,was #1
6,nautilus standard bunk bed,treva solid wood standard bunk bed by bedz king Kids Beds,was #43
7,carlen bunk bed,madelynn full over full metal bunk bed low bunk bed with ladder Kids Beds,was #24
8,holzer bunk bed Kids Beds,kemah twin solid wood standard bunk bed Kids Beds,was #34
9,meneses twin bunk bed,stig full over full solid wood standard bunk bed with trundle Kids Beds,was #41
10,ruthwell bunk bed Kids Beds,jenafir twin over full solid wood standard bunk bed Kids Beds,was #46


Read the `was #` column. Six of the new top ten came from below rank 10 — ranks 24, 29, 34, 41,
43 and 46 — where nobody would have seen them. Every one was *already retrieved*. That is the mechanism: the
search found them, the ordering buried them, and a model that reads query and product together
disagrees about which fifty belong at the top.

Nothing new entered the list. Nothing could.

Whether this new order is *better* is not something to decide by looking at it. That was the
lesson of [`01`](01_building_hybrid_search.ipynb), and it is why the next section exists.

## 4. Score it against the judgements

Four pipelines, the same 40 queries, the same `trec_eval` arithmetic as `02`:

- **hybrid** — the shortlist as Couchbase ordered it.
- **hybrid + department filter** — `02`'s filtered variant, the bet on a classifier.
- **hybrid → rerank** — the same shortlist, reordered by the cross-encoder.
- **hybrid + filter → rerank** — both, to see whether the bet still pays once a reranker follows.

`R@50` is in the table on purpose. Three of those four rows retrieve the same fifty products, so
three of them must post the same recall. A metric that cannot move is worth showing once,
because it is the clearest statement of what reranking does not do.

In [6]:
import time

from cbnb.eval import Run, compare, to_qrels

qrels = to_qrels(evalset.judgements())     # Exact=2, Partial=1, Irrelevant=0

class_to_department = (
    products[products.department != ""]
    .groupby("product_class").department
    .agg(lambda s: s.value_counts().idxmax())
)
query_department = evalset.queries.set_index("query_id").query_class.map(class_to_department)


def department_filter(query_id):
    department = query_department.get(query_id)
    return None if pd.isna(department) else search.TermQuery(department, field="department")


def run_strategy(name, *, filtered=False, rerank_fields=None):
    """Score one pipeline over every query, timing the whole pass."""
    results, started = {}, time.perf_counter()
    for row in evalset.queries.itertuples(index=False):
        hits = retrieve(row.query,
                        prefilter=department_filter(row.query_id) if filtered else None)
        if rerank_fields:
            scored = reranker.rank(row.query, hits, fields=rerank_fields)
            results[str(row.query_id)] = [(s.id, s.score) for s in scored]
        else:
            results[str(row.query_id)] = [(h.id, h.score) for h in hits]
    return Run(name, results), time.perf_counter() - started


runs, seconds = [], {}
for name, kwargs in {
    "hybrid": {},
    "hybrid + department filter": {"filtered": True},
    "hybrid -> rerank": {"rerank_fields": ["text"]},
    "hybrid + filter -> rerank": {"filtered": True, "rerank_fields": ["text"]},
}.items():
    run, elapsed = run_strategy(name, **kwargs)
    runs.append(run)
    seconds[name] = elapsed

scores = compare(runs, qrels, measures=["nDCG@10", "R@50", "RR"])
scores["seconds / 40 queries"] = pd.Series(seconds)
scores.style.format({"nDCG@10": "{:.3f}", "R@50": "{:.3f}", "RR": "{:.3f}",
                     "seconds / 40 queries": "{:.1f}"})

,nDCG@10,R@50,RR,seconds / 40 queries
hybrid,0.769,0.398,0.927,4.5
hybrid + department filter,0.773,0.403,0.927,4.6
hybrid -> rerank,0.766,0.398,0.967,7.4
hybrid + filter -> rerank,0.767,0.403,0.967,7.1


Two things happened, and only one was the plan.

**Recall did not move.** 0.398 for every unfiltered row, 0.403 for every filtered one. That is
the promise from section 1 arriving exactly as stated: the reranker reordered the fifty products
retrieval had already chosen, so the filtered rows differ only because the *filter* changed which
fifty those were.

**Reranking did not improve the ranking.** nDCG@10 went from 0.769 to 0.766 — a difference far
too small to mean anything, and in the wrong direction. RR improved (0.927 → 0.967), so the model
is better at getting *one* good product to the very top, but the ten results a person actually
sees are no better ordered than Couchbase had them, and the pass cost 60% more time.

This is the point where a demo quietly drops the reranker, or a blog post reports the RR column
and not the nDCG one. The more useful move is to ask what the model was working with.

## 5. What the reranker was given to read

Look again at what each candidate looked like to the model:

> `renan bunk bed`

That is the whole document. A cross-encoder's advantage is that it reads query and document
together — and there is almost nothing here to read. Three or four words, no material, no size,
no category, nothing that distinguishes a bunk bed for a dorm from one rated for an adult.

The catalogue does hold more. `category_path` is already in the index, already returned with
every hit, and already being ignored by the call above. Give it to the model and score the same
pipeline again.

In [7]:
richer, elapsed = run_strategy("hybrid -> rerank (+ category path)",
                               rerank_fields=["text", "category_path"])
runs.append(richer)
seconds["hybrid -> rerank (+ category path)"] = elapsed

scores = compare(runs, qrels, measures=["nDCG@10", "R@50", "RR", "P@10"])
scores["seconds / 40 queries"] = pd.Series(seconds)
scores.style.format({"nDCG@10": "{:.3f}", "R@50": "{:.3f}", "RR": "{:.3f}", "P@10": "{:.3f}",
                     "seconds / 40 queries": "{:.1f}"})

,nDCG@10,R@50,RR,P@10,seconds / 40 queries
hybrid,0.769,0.398,0.927,0.915,4.5
hybrid + department filter,0.773,0.403,0.927,0.927,4.6
hybrid -> rerank,0.766,0.398,0.967,0.912,7.4
hybrid + filter -> rerank,0.767,0.403,0.967,0.920,7.1
hybrid -> rerank (+ category path),0.794,0.398,0.983,0.922,7.8


Same model, same fifty candidates, same everything — except that each product now arrives as
`renan bunk bed | Furniture / Bedroom Furniture / Kids Beds` instead of `renan bunk bed`.

nDCG@10 **0.769 → 0.794**, RR **0.927 → 0.983**, P@10 0.915 → 0.922, for about three seconds more
per forty queries. The gain that the reranker was supposed to deliver was there all along, behind
a field the code was throwing away.

**A reranker cannot judge what it cannot read.** Its whole advantage over the bi-encoder is
attention to the query while reading the document, and a three-word title gives it nothing to
attend to. Before reaching for a bigger cross-encoder — the obvious and expensive next lever —
look at the string you are actually handing it.

It is also a warning about how this kind of result gets reported. "Reranking improved nDCG@10 by
0.025" is true here and would be equally true, with the opposite sign, if this notebook had stopped
one cell earlier.

## 6. How deep should the shortlist be?

Reranking fifty candidates costs fifty model passes; reranking ten costs ten. What do the extra
forty buy? Not recall — the shortlist is fixed before the reranker runs. What changes is how
much of it gets reordered, and how many distractors the model has the chance to promote.

One retrieval pass per query, deeper than before, then rerank the first *n* of it.

In [8]:
DEPTHS = [10, 25, 50, 100]

deep = {row.query_id: retrieve(row.query, k=max(DEPTHS))
        for row in evalset.queries.itertuples(index=False)}

sweep_runs = [Run("no rerank", {str(qid): [(h.id, h.score) for h in hits]
                                for qid, hits in deep.items()})]
for depth in DEPTHS:
    results = {}
    for row in evalset.queries.itertuples(index=False):
        scored = reranker.rank(row.query, deep[row.query_id][:depth],
                               fields=["text", "category_path"])
        # Candidates below the reranked depth keep their retrieval order, so every
        # row holds the same 100 products and only the ordering differs.
        tail = [(h.id, -rank) for rank, h in enumerate(deep[row.query_id][depth:], 1)]
        results[str(row.query_id)] = [(s.id, s.score) for s in scored] + tail
    sweep_runs.append(Run(f"rerank top {depth}", results))

compare(sweep_runs, qrels, measures=["nDCG@10", "RR", "R@100"]).style.format("{:.3f}")

,nDCG@10,RR,R@100
no rerank,0.769,0.927,0.619
rerank top 10,0.778,0.963,0.619
rerank top 25,0.797,0.958,0.619
rerank top 50,0.788,0.944,0.619
rerank top 100,0.798,0.983,0.619


`R@100` is identical down the column, because every row holds the same hundred products: the
reranked head, then the rest in retrieval order. Only the arrangement differs.

Reranking the first **ten** helps a little (0.769 → 0.778): the model can only reorder what is already on
the first screen. Going deeper is what pays, and past about twenty-five the differences
(0.797, 0.788, 0.798) are smaller than the noise this benchmark carries, while the cost keeps
climbing — reranking 100 candidates is four times the model passes of 25 for no measurable gain.

"Retrieve wide, rerank narrow" is the shape, but *wide* here means tens, not hundreds.

## 7. The filter was a bet. The reranker is not.

`02` ended on the department filter: it helped most queries a little and hurt one badly, because
*70s inspired furniture* is labelled `Wall Art` in WANDS, so the filter excluded the furniture
department for a query with "furniture" in it. The right products were not ranked badly. They
were absent.

A reranker facing the same uncertainty cannot do that. Its worst case is a product ranked below
where it belongs — still retrieved, still there for anything downstream to recover. The two
columns on the right are the same queries under both bets.

In [9]:
from cbnb.eval import score_per_query

keep = {"hybrid", "hybrid + department filter", "hybrid -> rerank (+ category path)"}
per_query = pd.DataFrame({run.name: score_per_query(run, qrels, "nDCG@10")
                          for run in runs if run.name in keep})
per_query["rerank vs hybrid"] = per_query["hybrid -> rerank (+ category path)"] - per_query["hybrid"]
per_query["filter vs hybrid"] = per_query["hybrid + department filter"] - per_query["hybrid"]
per_query = per_query.join(evalset.queries.set_index(evalset.queries.query_id.astype(str))["query"])

print(f"reranking helped {(per_query['rerank vs hybrid'] > 0).sum()} of {len(per_query)} queries, "
      f"hurt {(per_query['rerank vs hybrid'] < 0).sum()}, "
      f"worst case {per_query['rerank vs hybrid'].min():+.3f}")
print(f"the filter helped {(per_query['filter vs hybrid'] > 0).sum()}, "
      f"hurt {(per_query['filter vs hybrid'] < 0).sum()}, "
      f"worst case {per_query['filter vs hybrid'].min():+.3f}")

extremes = per_query.sort_values("rerank vs hybrid")
columns = ["query", "hybrid", "hybrid -> rerank (+ category path)", "rerank vs hybrid", "filter vs hybrid"]
pd.concat([extremes.head(4), extremes.tail(4)])[columns].style.format(
    {"hybrid": "{:.3f}", "hybrid -> rerank (+ category path)": "{:.3f}",
     "rerank vs hybrid": "{:+.3f}", "filter vs hybrid": "{:+.3f}"}
)

reranking helped 18 of 40 queries, hurt 10, worst case -0.233
the filter helped 10, hurt 4, worst case -0.123


,query,hybrid,hybrid -> rerank (+ category path),rerank vs hybrid,filter vs hybrid
410,pool floats,1.000,0.767,-0.233,+0.000
178,almost heaven sauna,0.562,0.451,-0.110,+0.000
460,small wardrobe grey,0.321,0.213,-0.108,+0.035
256,high weight capacity bunk beds,0.888,0.793,-0.095,+0.000
220,sheets for twinxl,0.863,1.000,+0.137,+0.000
472,window wall accent,0.776,0.957,+0.181,+0.000
93,outdoor sectional dining,0.489,0.679,+0.191,+0.074
145,liberty hardware francisco,0.000,0.607,+0.607,+0.064


The counts tell the story the table detail fills in: reranking helped 18 of 40 queries and hurt 10;
the filter helped 10 and hurt 4, leaving most queries untouched because most of its departments
were already right.

**The bottom rows are the case for reranking.** *liberty hardware francisco* goes from 0.000 to
0.607 — a query where the lexical and vector halves both floundered and a model that reads the
whole candidate did not.

**The top rows are its cost, and they are survivable.** *pool floats* falls from a perfect 1.000
to 0.767: the reranker demoted products a human had judged `Exact`. They are still in the fifty.
A later stage can still find them, and `R@50` never moved. Compare that with the filter's failure
mode from `02` — the right products excluded from the shortlist entirely, unrecoverable by
anything downstream.

That is the asymmetry worth taking away. Both are bets on a model's judgement. One bets ordering,
which anything downstream can revisit; the other bets membership, which nothing can.

## 8. Does it survive a re-run?

`02` found these scores move between runs: approximate retrieval does not return exactly the
same candidates twice. The reranker itself is deterministic, so any drift here arrives through
the shortlist. Run the two ends of the comparison again and look at the spread before believing
any gap.

In [10]:
repeat = {}
for name, kwargs in {"hybrid": {},
                     "hybrid -> rerank (+ category path)": {"rerank_fields": ["text", "category_path"]}}.items():
    run, _ = run_strategy(name, **kwargs)
    repeat[name] = compare([run], qrels, measures=["nDCG@10", "R@50"]).loc[name]

second = pd.DataFrame(repeat).T
pd.DataFrame({
    "run 1 nDCG@10": scores.loc[second.index, "nDCG@10"],
    "run 2 nDCG@10": second["nDCG@10"],
    "run 1 R@50": scores.loc[second.index, "R@50"],
    "run 2 R@50": second["R@50"],
}).style.format("{:.3f}")

,run 1 nDCG@10,run 2 nDCG@10,run 1 R@50,run 2 R@50
hybrid,0.769,0.769,0.398,0.398
hybrid -> rerank (+ category path),0.794,0.794,0.398,0.398


Identical to three decimals, both measures, both pipelines. Two passes inside one session hit the
same index in the same state and got the same fifty candidates back; the drift `02` reported
appears between separate runs, where the index has been rebuilt in between.

Worth knowing for reading the numbers above: `02` saw movement of a few thousandths between runs.
The +0.025 from giving the reranker the category path is an order of magnitude larger than that.
The differences among reranking depths, at 0.01 or less, are not.

## Where to take this

- **Fix the document before fixing the model.** The largest gain here came from a field that was
  already in the index. A bigger cross-encoder is the obvious next lever and the expensive one;
  check what the model can actually see first.
- **Cache by (query, product).** A cross-encoder score does not change between runs for the same
  pair, and Couchbase is already in the request path.
- **Rerank for the model, not the screen.** When these results become RAG context, ordering
  decides what survives a trim to fit and what lands in the middle of the window, where models
  attend least. [`flows/01`](../flows/01_rag_that_you_can_trust.ipynb) retrieves four chunks per
  question with no reranking at all — an obvious thing to measure next.
- **Try a bigger cross-encoder.** `ms-marco-MiniLM-L-6-v2` is the small one in that family. The
  larger models cost more per pair and usually rank better; whether that is worth it is exactly
  the sort of question this notebook's table answers.

### On the word "reranking"

Couchbase's docs use `rerank` for something else. A
[Hyperscale Vector index](https://docs.couchbase.com/server/current/vector-index/hyperscale-reranking.html)
searches *quantised* vectors, and its `rerank` argument re-scores the survivors against the
full-precision vectors it kept. Same embedding model, same notion of relevance, better
arithmetic — it recovers precision that compression gave away. No second model, and nothing
re-reads the query.

The Search index this notebook uses is a different service and does not quantise: its scores
match a full-precision dot product computed by hand. The two ideas never collide here, but the
word does.

> **On Couchbase AI Data Plane** — the cross-encoder runs on your machine and its scores are
> recomputed from scratch on every run. Capella's Model Service hosts models behind an
> OpenAI-compatible endpoint, so an application does not carry its own inference; it is
> Enterprise Support only, which is why this notebook ships a local model instead.

In [11]:
# Everything here lives in the scope `retrieval/01` created, and `02` shares it.
# Uncomment to remove it.
# from cbnb.couchbase_io import drop_demo_data
# drop_demo_data(cluster, BUCKET, SCOPE)